In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from collections import defaultdict
import sys
sys.path.append("../../")
from src.utils import utils as ut

do_graph = ut.load_do_graph()
doid_to_term = {
    node: data["name"] for node, data in do_graph.nodes(data=True) if "name" in data
}


In [15]:
doid_to_icd = dict()
for node, data in do_graph.nodes(data=True):
    for ref in data.get("xref", []):
        if ref.startswith("ICD9CM:"):
            icd = ref.split(":")[1]
            doid_to_icd[node] = icd

In [20]:
len(doid_to_icd)
icd_to_doid = {v: k for k, v in doid_to_icd.items()}    

In [ ]:
import pandas as pd

In [19]:

# =========================
# CONFIG
# =========================
PDN_TSV   = "/home/ddalton/Downloads/allnet5/AllNet5.tsv"   # <-- your file from the message
TSV_SEP   = "\t"
DOID_OBO  = "doid.obo"        # <-- download DO OBO and set this path
ALPHA     = 0.05              # significance threshold for pval.adj or pval
REQUIRE_RR_GT_1 = True        # set False if you want significance alone to define comorbidity

# =========================
# 1) Load PDN TSV
# =========================
df = pd.read_csv(PDN_TSV, sep=TSV_SEP, dtype=str).rename(columns=lambda c: c.strip())
# Coerce numeric columns
for col in ["n1","n2","n.common","RR","RR.low","RR.high","pval","pval.adj"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Normalize ICD-9 code strings: keep dotted and non-dotted forms available for matching
def norm_icd9(code: str):
    if pd.isna(code):
        return None
    s = str(code).strip()
    # sometimes loaders turn codes like 0.02 into '0.02' - keep as-is and also a non-dotted variant
    return s

df["icd9_1"] = df.get("#icd9-1", df.get("icd9-1", df.get("icd9_1"))).apply(norm_icd9)
df["icd9_2"] = df.get("icd9-2", df.get("icd9_2")).apply(norm_icd9)

# =========================
# 2) Parse DOID OBO: build ICD9CM -> (doid, name) map from xrefs
# =========================



In [23]:
df["icd9_1"]

0          000.02
1          000.02
2          000.02
3          000.02
4          000.02
            ...  
6088548     999.6
6088549     999.6
6088550     999.7
6088551     999.8
6088552     999.9
Name: icd9_1, Length: 6088553, dtype: object

In [24]:



df["doid_1"] = df["icd9_1"].map(icd_to_doid)
df["doid_2"] = df["icd9_2"].map(icd_to_doid)
df["name_1"] = df["doid_1"].map(doid_to_term)
df["name_2"] = df["doid_2"].map(doid_to_term)


In [30]:
df.dropna(subset=["doid_1", "doid_2"], how="any", inplace=True)

In [ ]:
df_filt = df[df["pval.adj"]<0.05]

,#icd9-1,icd9-2,n1,n2,n.common,RR,RR.low,RR.high,pval,pval.adj,icd9_1,icd9_2,doid_1,doid_2,name_1,name_2
1320,002.0,162.9,111,82175,1,1.429494,0.106174,19.246325,0.000100,0.028608,002.0,162.9,DOID:13258,DOID:1325,typhoid fever,bronchus cancer
1324,002.0,185,111,392281,2,0.598900,0.095261,3.765262,-0.000206,-0.129092,002.0,185,DOID:13258,DOID:10283,typhoid fever,prostate cancer
1343,002.0,244.9,111,628101,2,0.374044,0.059495,2.351598,-0.000411,-0.325621,002.0,244.9,DOID:13258,DOID:1459,typhoid fever,hypothyroidism
1403,002.0,300.4,111,95068,1,1.235627,0.091774,16.636163,0.000059,0.018166,002.0,300.4,DOID:13258,DOID:12139,typhoid fever,dysthymic disorder
1411,002.0,331.0,111,229250,1,0.512404,0.038058,6.898873,-0.000190,-0.091125,002.0,331.0,DOID:13258,DOID:10652,typhoid fever,Alzheimer's disease
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5742999,757.0,791.0,662,11101,1,1.774291,0.131783,23.888586,0.000161,0.016967,757.0,791.0,DOID:0050580,DOID:576,hereditary lymphedema,proteinuria
5743935,758.0,780.57,549,9728,1,2.441456,0.181336,32.871131,0.000256,0.025205,758.0,780.57,DOID:14250,DOID:0050847,Down syndrome,sleep apnea
5744968,759.3,780.57,793,9728,1,1.690239,0.125540,22.756935,0.000147,0.014506,759.3,780.57,DOID:758,DOID:0050847,situs inversus,sleep apnea
5744978,759.3,782.1,793,22099,1,0.744045,0.055263,10.017623,-0.000082,-0.012226,759.3,782.1,DOID:758,DOID:0050486,situs inversus,exanthem


In [37]:
import scanpy as sc
data_path = "/aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-09-12-01/data.h5ad"

# 2. Load Data
adata = sc.read_h5ad(data_path, backed='r')

In [ ]:
doid_vals = list(adata.obs["doid_id"].unique())
df_filt = df[df["pval.adj"]<0.05]
df_filt.query("doid_1 in @doid_vals and doid_2 in @doid_vals")

,#icd9-1,icd9-2,n1,n2,n.common,RR,RR.low,RR.high,pval,pval.adj,icd9_1,icd9_2,doid_1,doid_2,name_1,name_2
417632,061,428.0,9,2343516,1,0.618208,0.045917,8.323386,-0.000148,-0.227297,061,428.0,DOID:12205,DOID:6000,dengue disease,congestive heart failure
441378,076.1,427.31,24,1806969,3,0.901995,0.201039,4.046953,-0.000053,-0.071689,076.1,427.31,DOID:11265,DOID:0060224,trachoma,atrial fibrillation
469156,083.0,428.0,26,2343516,4,0.855980,0.233282,3.140843,-0.000095,-0.145732,083.0,428.0,DOID:11100,DOID:6000,Q fever,congestive heart failure
469157,083.0,429.2,26,384087,1,1.305696,0.096979,17.579542,0.000075,0.046607,083.0,429.2,DOID:11100,DOID:1287,Q fever,cardiovascular system disease
469180,083.0,530.1,26,425051,1,1.179860,0.087632,15.885325,0.000047,0.030396,083.0,530.1,DOID:11100,DOID:11963,Q fever,esophagitis
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5398671,710.1,746.3,7661,458,1,3.716156,0.276012,50.033360,0.000390,0.034159,710.1,746.3,DOID:418,DOID:1712,systemic scleroderma,aortic valve stenosis
5423968,714.0,746.3,175906,458,4,0.647379,0.176431,2.375425,-0.000244,-0.102502,714.0,746.3,DOID:7148,DOID:1712,rheumatoid arthritis,aortic valve stenosis
5424135,714.0,759.82,175906,220,1,0.336931,0.025025,4.536356,-0.000319,-0.133585,714.0,759.82,DOID:7148,DOID:14323,rheumatoid arthritis,Marfan syndrome
5550176,720.0,746.3,6375,458,1,4.465799,0.331691,60.126369,0.000454,0.036267,720.0,746.3,DOID:7147,DOID:1712,ankylosing spondylitis,aortic valve stenosis
